# Lab Day 1 — Forest CoverType

Notebook chạy từ `submission_2A202602759/code/`. Tất cả cấu hình chọn bằng validation; eval chỉ dùng sau khi chọn cấu hình cuối cùng. Kết quả của các ô phải được giữ lại trong file nộp.

In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math
from pathlib import Path
import numpy as np, torch
# Matplotlib được dùng nếu có; plots.py dùng Pillow khi môi trường CPU thiếu Matplotlib.

working_dir = Path.cwd().resolve()
candidate_dirs = [working_dir, *working_dir.glob("submission_*/code"),
                  *working_dir.glob("*/submission_*/code")]
CODE_DIR = Path(os.environ.get("LAB_CODE_DIR", next((str(p) for p in candidate_dirs
    if p.name == "code" and p.parent.name.startswith("submission_")), ""))).resolve()
if not (CODE_DIR / "data.py").exists():
    raise FileNotFoundError("Không tìm thấy submission_<MSSV>/code; đặt LAB_CODE_DIR trước khi chạy.")
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
REPO_ROOT = CODE_DIR.parents[1]
OUT_DIR = CODE_DIR.parent
for folder in (OUT_DIR / "figures", OUT_DIR / "results"):
    folder.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval, compute_loss
from plots import plot_run, plot_compare, plot_health
from results_table import save_result, load_results, to_row, write_xlsx, write_report

PyTorch: 2.14.0+cpu | device: cpu


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [3]:
# Part 0: chỉ tạo lại processed data nếu chưa có.
if not (REPO_ROOT / "data/processed/train.npz").exists() or not (REPO_ROOT / "data/processed/eval.npz").exists():
    subprocess.run([sys.executable, str(REPO_ROOT / "scripts/split_data.py")], cwd=REPO_ROOT, check=True)
data = prepare_data(device, val_fraction=0.2, seed=42,
                    processed_dir=str(REPO_ROOT / "data/processed"))
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962
assert len(data["X_eval"]) == 116_203
mean10 = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
std10 = data["X_tr"][:, :10].std(dim=0, unbiased=False).cpu().numpy()
print("mean 10 numeric:", np.round(mean10, 4))
print("std 10 numeric:", np.round(std10, 4))
assert np.allclose(mean10, 0, atol=1e-3) and np.allclose(std10, 1, atol=1e-3)
majority = int(torch.bincount(data["y_tr"], minlength=7).argmax())
majority_acc = (data["y_val"] == majority).float().mean().item()
print("Majority class:", majority, "| val accuracy:", majority_acc)

train=371,847; val=92,962; eval=116,203; majority-val-accuracy=0.4876
mean 10 numeric: [-0. -0.  0.  0.  0.  0. -0. -0. -0.  0.]
std 10 numeric: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Majority class: 1 | val accuracy: 0.4875970780849457


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [5]:
# Part 1: kiểm tra mô hình, loss ban đầu và gradient.
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879
logits = model(data["X_tr"][:8])
print("Input:", tuple(data["X_tr"][:8].shape), "| logits:", tuple(logits.shape))
assert logits.shape == (8, 7)
step0_loss = evaluate(model, data["X_val"], data["y_val"])["loss"]
print(f"Step-0 loss={step0_loss:.4f}; ln(7)={math.log(7):.4f}; gap={step0_loss-math.log(7):+.4f}")

small_x, small_y = data["X_tr"][:20], data["y_tr"][:20]
small_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
small_opt = torch.optim.Adam(small_model.parameters(), lr=0.01)
small_losses = []
for step in range(500):
    small_opt.zero_grad(set_to_none=True)
    loss = compute_loss(small_model(small_x), small_y, "ce")
    loss.backward()
    small_opt.step()
    small_losses.append(float(loss.item()))
small_acc = (predict(small_model, small_x) == small_y).float().mean().item()
print(f"Overfit 20 samples: final loss={small_losses[-1]:.6f}; accuracy={small_acc:.3f}")
plot_health(small_losses, OUT_DIR / "figures/compare_health_overfit20.png")

model.zero_grad(set_to_none=True)
compute_loss(model(data["X_tr"][:512]), data["y_tr"][:512], "ce").backward()
gradient_norms = {name: float(param.grad.norm()) if param.grad is not None else None
                  for name, param in model.named_parameters()}
print("Gradient norms:", gradient_norms)
assert all(value is not None and value > 0 for value in gradient_norms.values())

Input: (8, 54) | logits: (8, 7)
Step-0 loss=1.9655; ln(7)=1.9459; gap=+0.0196
Overfit 20 samples: final loss=0.000001; accuracy=1.000
Gradient norms: {'net.0.weight': 0.04406192526221275, 'net.0.bias': 0.029319457709789276, 'net.2.weight': 0.17308413982391357, 'net.2.bias': 0.03812722861766815, 'net.4.weight': 1.7037497758865356, 'net.4.bias': 0.5023939609527588}


**Nhận xét Part 1:** Ô trên in loss đo được và độ lệch so với ln 7, cùng loss/accuracy cuối của phép thử 20 mẫu và chuẩn gradient từng tham số. He được dùng ở lớp ẩn; trọng số lớp logit được nhân 0,1 sau khi khởi tạo He để giảm độ lớn logit bước 0 theo gợi ý trong GUIDE. Nếu 20 mẫu không đạt accuracy 100%, cần chẩn đoán vòng huấn luyện trước khi chạy dài.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [8]:
# Part 2: chọn learning rate bằng validation, rồi đo nhiễu hai seed baseline.
# Hai lần dò LR dùng 5 epoch để tiết kiệm thời gian, không ghi vào bảng thí nghiệm 20 epoch.
lr_candidates = (0.01, 0.05, 0.1)
lr_probe = {}
for lr in lr_candidates:
    probe_cfg = {**DEFAULT_CFG, "exp_id": f"probe-lr{lr}", "lr": lr, "epochs": 5, "seed": 1}
    probe = run_experiment(probe_cfg, data)
    lr_probe[lr] = probe["summary"]["val_macro_f1"]
print("LR probe on VAL:", lr_probe)
chosen_lr = max(lr_probe, key=lambda lr: -1 if lr_probe[lr] is None else lr_probe[lr])
cfg = {**DEFAULT_CFG, "lr": chosen_lr}
print("Chosen baseline LR from VAL:", chosen_lr)

baseline_results = []
for seed in (1, 2):
    run_cfg = {**cfg, "seed": seed, "exp_id": f"base-s{seed}"}
    result = run_experiment(run_cfg, data)
    baseline_results.append(result)
    save_result(result, OUT_DIR / "results")
    plot_run(result, OUT_DIR / f"figures/{run_cfg['exp_id']}.png")
    print(run_cfg["exp_id"], result["summary"])
baseline_f1 = np.array([r["summary"]["val_macro_f1"] for r in baseline_results])
baseline_acc = np.array([r["summary"]["val_acc"] for r in baseline_results])
noise_2sigma = 2 * baseline_f1.std(ddof=1)
print(f"Baseline val macro-F1={baseline_f1.mean():.4f} ± {baseline_f1.std(ddof=1):.4f}; "
      f"val acc={baseline_acc.mean():.4f} ± {baseline_acc.std(ddof=1):.4f}; "
      f"2 sigma={noise_2sigma:.4f}; majority val acc={majority_acc:.4f}")

LR probe on VAL: {0.01: 0.6122960705532454, 0.05: 0.7328372965023269, 0.1: 0.762152181717663}
Chosen baseline LR from VAL: 0.1
base-s1 {'step0_loss': 1.9655427713407092, 'best_val_loss': 0.2331244187865825, 'best_epoch': 19, 'final_train_loss': 0.2115228153746387, 'final_val_loss': 0.23396918994265634, 'val_acc': 0.9062950452873217, 'val_macro_f1': 0.8555793662292974, 'time_per_epoch_s': 2.686531045001175, 'peak_mem_MB': None, 'diverged': False}
base-s2 {'step0_loss': 1.9370584185692137, 'best_val_loss': 0.23084731346250686, 'best_epoch': 20, 'final_train_loss': 0.20804495372219833, 'final_val_loss': 0.23084731346250686, 'val_acc': 0.907542866977905, 'val_macro_f1': 0.8594958604217361, 'time_per_epoch_s': 3.0660293149951032, 'peak_mem_MB': None, 'diverged': False}
Baseline val macro-F1=0.8575 ± 0.0028; val acc=0.9069 ± 0.0009; 2 sigma=0.0055; majority val acc=0.4876


**Nhận xét baseline:** So sánh đường cong `figures/base-s1.png` và `figures/base-s2.png`. Số liệu val macro-F1, accuracy và ngưỡng nhiễu 2σ được in ở ô trên. Nếu chênh lệch giữa các cấu hình nhỏ hơn 2σ, cần ghi là chưa đủ bằng chứng.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Các thí nghiệm Part 3

Dự đoán được khai báo trước khi chạy trong `experiment_specs` ở ô tiếp theo. Mỗi cấu hình đổi một yếu tố so với `base-s1`, cùng split, seed và 20 epoch. So sánh CE/MSE bằng val macro-F1 vì hai loss khác thang đo. Optimizer có hai mức LR cho mỗi loại để so công bằng.

In [12]:
# Dự đoán trước, mỗi lần chỉ thay một mục của baseline.
experiment_specs = [
    ("loss-mse", "loss", {"loss": "mse"}, "MSE trên one-hot có thể cho macro-F1 thấp hơn CE vì gradient khác."),
    ("opt-adam-1e3", "optimizer", {"optimizer": "adam", "lr": 0.001}, "Adam 0.001 có thể hội tụ nhanh hơn SGD+momentum."),
    ("opt-adam-3e3", "optimizer", {"optimizer": "adam", "lr": 0.003}, "Adam 0.003 có thể nhanh hơn 0.001 nhưng dao động hơn."),
    ("opt-sgdm-half", "optimizer", {"lr": chosen_lr / 2}, "Giảm LR của SGD+momentum có thể chậm hội tụ hơn."),
    ("wide", "hparam", {"hidden": (512, 256)}, "Mạng rộng có thể tăng val macro-F1 nhưng tốn thời gian hơn."),
    ("dropout-02", "dropout", {"dropout": 0.2}, "Dropout giúp nếu train-val gap cho thấy quá khớp; nếu chưa, có thể giảm điểm."),
    ("clip-1", "clipping", {"clip_norm": 1.0}, "Clip 1.0 có thể ít tác dụng khi norm vốn nhỏ."),
    ("init-xavier", "init", {"init": "xavier"}, "Xavier có thể thay đổi kích hoạt và tốc độ hội tụ so với He."),
]
if device == "cuda":
    experiment_specs.append(("amp-fp16", "amp", {"precision": "fp16"},
                             "FP16 có thể giảm bộ nhớ và thời gian, nhưng MLP nhỏ có thể không nhanh hơn."))

experiment_results = []
for exp_id, group, changes, prediction in experiment_specs:
    print(f"[{exp_id}] Dự đoán trước: {prediction}")
    exp_cfg = {**cfg, "exp_id": exp_id, "group": group,
               "description": prediction, "seed": 1, **changes}
    result = run_experiment(exp_cfg, data)
    experiment_results.append(result)
    save_result(result, OUT_DIR / "results")
    plot_run(result, OUT_DIR / f"figures/{exp_id}.png")
    value = result["summary"]["val_macro_f1"]
    base_value = baseline_results[0]["summary"]["val_macro_f1"]
    if value is None:
        print("Kết quả: phân kỳ hoặc không có epoch hợp lệ; xem history và giảm LR.")
    else:
        delta = value - base_value
        print(f"val macro-F1={value:.4f}; Δ so với base-s1={delta:+.4f}; "
              f"vượt 2σ={abs(delta)>noise_2sigma}; best epoch={result['summary']['best_epoch']}")

[loss-mse] Dự đoán trước: MSE trên one-hot có thể cho macro-F1 thấp hơn CE vì gradient khác.
val macro-F1=0.7209; Δ so với base-s1=-0.1347; vượt 2σ=True; best epoch=20
[opt-adam-1e3] Dự đoán trước: Adam 0.001 có thể hội tụ nhanh hơn SGD+momentum.
val macro-F1=0.8516; Δ so với base-s1=-0.0039; vượt 2σ=False; best epoch=19
[opt-adam-3e3] Dự đoán trước: Adam 0.003 có thể nhanh hơn 0.001 nhưng dao động hơn.
val macro-F1=0.8766; Δ so với base-s1=+0.0210; vượt 2σ=True; best epoch=18
[opt-sgdm-half] Dự đoán trước: Giảm LR của SGD+momentum có thể chậm hội tụ hơn.
val macro-F1=0.8283; Δ so với base-s1=-0.0273; vượt 2σ=True; best epoch=17
[wide] Dự đoán trước: Mạng rộng có thể tăng val macro-F1 nhưng tốn thời gian hơn.
val macro-F1=0.8758; Δ so với base-s1=+0.0202; vượt 2σ=True; best epoch=20
[dropout-02] Dự đoán trước: Dropout giúp nếu train-val gap cho thấy quá khớp; nếu chưa, có thể giảm điểm.
val macro-F1=0.8167; Δ so với base-s1=-0.0388; vượt 2σ=True; best epoch=20
[clip-1] Dự đoán trước: C

**Đối chiếu sau khi chạy:** Ô trên in chênh lệch val macro-F1 và so với 2σ cho từng `exp_id`. Đọc thêm ảnh riêng để xét tốc độ giảm loss và grad norm. Với clipping, dùng đường `grad_norm` trước khi cắt để xem ngưỡng 1.0 có kích hoạt. Với dropout, xét train–val loss gap. Với mixed precision, so `time_per_epoch_s`, `peak_mem_MB` và val macro-F1.

In [14]:
# Ảnh so sánh cùng chủ đề, chỉ cho nhóm có từ hai lần chạy.
for group in sorted({r["cfg"]["group"] for r in experiment_results}):
    group_results = [r for r in experiment_results if r["cfg"]["group"] == group]
    if len(group_results) >= 2:
        plot_compare(group_results, "val_macro_f1", OUT_DIR / f"figures/compare_{group}.png",
                     f"Validation macro-F1 — {group}")

# Khởi tạo: đo độ lệch chuẩn sau mỗi Linear/ReLU ở bước 0 trên cùng lô val.
init_diagnostics = {}
for init_name in ("he", "xavier", "normal", "zeros"):
    set_seed(1)
    initial_model = MLP(hidden=(256, 128), init=init_name).to(device)
    init_diagnostics[init_name] = {"activation_std": activation_stats(initial_model, data["X_val"][:512]),
                                   "step0_val_loss": evaluate(initial_model, data["X_val"], data["y_val"])["loss"]}
print("Step-0 initialization diagnostics:", init_diagnostics)
(OUT_DIR / "results/initialization_step0.json").write_text(json.dumps(init_diagnostics, indent=2), encoding="utf-8")

Step-0 initialization diagnostics: {'he': {'activation_std': [0.6662027835845947, 0.3951933979988098, 0.6463822722434998, 0.3700721263885498, 0.059326160699129105], 'step0_val_loss': 1.9655427713407092}, 'xavier': {'activation_std': [0.2780497074127197, 0.16493988037109375, 0.2202722579240799, 0.12611210346221924, 0.196858748793602], 'step0_val_loss': 2.022176268722368}, 'normal': {'activation_std': [0.03461690992116928, 0.020534850656986237, 0.0037999360356479883, 0.0021755709312856197, 0.0002790121070574969], 'step0_val_loss': 1.9459956483395098}, 'zeros': {'activation_std': [0.0, 0.0, 0.0, 0.0, 0.0], 'step0_val_loss': 1.9459096179336046}}


## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [16]:
# Part 4: chọn cấu hình cuối cùng bằng VAL, rồi mới chấm EVAL.
eligible = [baseline_results[0]] + [r for r in experiment_results if r["summary"]["val_macro_f1"] is not None]
result_final = max(eligible, key=lambda r: r["summary"]["val_macro_f1"])
cfg_final = result_final["cfg"]
print("Chosen from VAL:", cfg_final["exp_id"], result_final["summary"]["val_macro_f1"])

baseline_pred = OUT_DIR / "results/baseline_predictions_eval.csv"
final_eval(baseline_results[0]["cfg"], baseline_results[0], data, baseline_pred)
baseline_eval_json = OUT_DIR / "results/baseline_eval_result.json"
subprocess.run([sys.executable, str(REPO_ROOT / "scripts/evaluate.py"),
                "--pred", str(baseline_pred), "--out", str(baseline_eval_json)],
               cwd=REPO_ROOT, check=True, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
baseline_eval = json.loads(baseline_eval_json.read_text(encoding="utf-8"))

prediction_path = OUT_DIR / "predictions_eval.csv"
final_eval(cfg_final, result_final, data, prediction_path)
eval_path = OUT_DIR / "eval_result.json"
subprocess.run([sys.executable, str(REPO_ROOT / "scripts/evaluate.py"),
                "--pred", str(prediction_path), "--out", str(eval_path)],
               cwd=REPO_ROOT, check=True, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
final_scores = json.loads(eval_path.read_text(encoding="utf-8"))
print("Baseline eval:", baseline_eval["accuracy"], baseline_eval["macro_f1"])
print("Final eval:", final_scores["accuracy"], final_scores["macro_f1"])

Chosen from VAL: opt-adam-3e3 0.8766143428441703
Baseline eval: 0.9053552834264176 0.8588917159340463
Final eval: 0.9124377167543006 0.8740767967982913


In [17]:
# Bảng từng lớp và ma trận nhầm lẫn lấy từ evaluate.py.
for cls in final_scores["per_class"]:
    print(cls)
cm = np.array(final_scores["confusion_matrix"])
print("Confusion matrix (rows=true, cols=pred):", cm)
hardest = min(final_scores["per_class"], key=lambda row: row["f1"])
hardest_id = hardest["cls"]
confused_row = cm[hardest_id].copy(); confused_row[hardest_id] = 0
print("Hardest class:", hardest_id, "F1:", hardest["f1"],
      "most confused with:", int(confused_row.argmax()), "support:", hardest["support"])

{'cls': 0, 'support': 42368, 'precision': 0.9052792475014697, 'recall': 0.9086338746223565, 'f1': 0.9069534590602287}
{'cls': 1, 'support': 56661, 'precision': 0.9252904094064315, 'recall': 0.9222039851043928, 'f1': 0.9237446191650536}
{'cls': 2, 'support': 7151, 'precision': 0.9055096791661026, 'recall': 0.9353936512375891, 'f1': 0.9202091071674233}
{'cls': 3, 'support': 549, 'precision': 0.891832229580574, 'recall': 0.7358834244080146, 'f1': 0.8063872255489023}
{'cls': 4, 'support': 1899, 'precision': 0.7970244420828906, 'recall': 0.7898894154818326, 'f1': 0.7934408886537953}
{'cls': 5, 'support': 3473, 'precision': 0.8656987295825771, 'recall': 0.824071408004607, 'f1': 0.8443723263018145}
{'cls': 6, 'support': 4102, 'precision': 0.915031115366204, 'recall': 0.931984397854705, 'f1': 0.9234299516908213}
Confusion matrix (rows=true, cols=pred): [[38497  3542     2     0    39     5   283]
 [ 3730 52253   139     0   328   139    72]
 [    0   178  6689    33    10   241     0]
 [    0 

**Phân tích lỗi:** Ô trên in support, precision, recall và F1 của cả 7 lớp, ma trận nhầm lẫn và lớp khó nhất. Đối chiếu support của lớp khó với lớp hay bị nhầm; mất cân bằng mẫu là một giả thuyết, không phải kết luận nhân quả nếu chưa thử thêm thí nghiệm.

In [19]:
# Bảng kết quả: dữ liệu trong JSON từng run, điểm eval chỉ cho baseline và cấu hình cuối.
results = load_results(OUT_DIR / "results")
eval_by_id = {baseline_results[0]["cfg"]["exp_id"]: baseline_eval,
              result_final["cfg"]["exp_id"]: final_scores}
rows = [to_row(result, eval_by_id.get(result["cfg"]["exp_id"])) for result in results]
for row in rows:
    if row["exp_id"] == "base-s1":
        row["notes"] = (f"majority_val_acc={majority_acc:.6f}; overfit20_loss={small_losses[-1]:.8g}; "
                        f"overfit20_acc={small_acc:.4f}; gradients_ok={all(v is not None and v > 0 for v in gradient_norms.values())}; "
                        f"init_step0={json.dumps(init_diagnostics, ensure_ascii=False)}")
    elif row["exp_id"] == "init-xavier":
        row["notes"] = f"activation_std={init_diagnostics['xavier']['activation_std']}"
write_xlsx(rows, REPO_ROOT / "templates/experiment_table_template.xlsx",
           OUT_DIR / "experiments.xlsx")
health = {"device": device, "torch_version": torch.__version__, "majority_acc": majority_acc,
          "n_params": count_params(model), "logits_shape": tuple(logits.shape),
          "step0_loss": step0_loss, "small_loss": small_losses[-1],
          "small_acc": small_acc, "gradients_ok": all(v is not None and v > 0 for v in gradient_norms.values()),
          "init_diagnostics": init_diagnostics}
write_report(rows, baseline_eval, final_scores, cfg_final["exp_id"], health, OUT_DIR / "REPORT.md")
print(f"Wrote {len(rows)} experiment rows and REPORT.md")

Wrote 10 experiment rows and REPORT.md
